    1. Ordenar datos por peso (fnlwgt) y crear dos muestras distintas:
       - Muestra 1: Primeros 2500 registros (peso más alto)
       - Muestra 2: Últimos 2500 registros (peso más bajo)

    2. Calcular métricas de calidad: Silhouette y Davies-Bouldin.

    3. Comparar resultados entre K-means (numérico) y Agglomerative (Gower).

    4. Analizar diferencias entre muestras de alto vs bajo peso.

    5. Documentar similitudes y diferencias

## Paso 1: Preparación de datos y creación de muestras por peso

**Estrategia de muestreo:**
1. Ordenar todo el dataset por `fnlwgt` (de mayor a menor)
2. Muestra 1: Primeros 2500 registros (peso más alto)
3. Muestra 2: Últimos 2500 registros (peso más bajo)

**Objetivo:** Analizar si el clustering se comporta diferente en personas con mayor vs menor peso.


In [1]:
import sys
sys.path.append('..')
from utils import cargar_dataset_adult
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Configuración
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

# Cargar dataset
df = cargar_dataset_adult()

print("="*70)
print("EJERCICIO 6: EVALUACIÓN DE CLUSTERS POR PESO")
print("="*70)
print(f"\nDataset completo: {len(df):,} observaciones")

# ============================================================================
# ESTRATEGIA DE MUESTREO POR PESO
# ============================================================================
# 1. Ordenar por fnlwgt (de mayor a menor)
# 2. Muestra 1: Primeros 2500 (peso más alto)
# 3. Muestra 2: Últimos 2500 (peso más bajo)
# ============================================================================

print("\n" + "-"*70)
print("CREANDO MUESTRAS POR PESO (fnlwgt)")
print("-"*70)

# Ordenar por fnlwgt (de mayor a menor)
df_sorted = df.sort_values('fnlwgt', ascending=False).reset_index(drop=True)

# Crear las dos muestras
sample_size = 2500
df_high_weight = df_sorted.head(sample_size).copy()  # Primeros 2500 (peso alto)
df_low_weight = df_sorted.tail(sample_size).copy()   # Últimos 2500 (peso bajo)

print(f"\nMuestra 1 (PESO ALTO):")
print(f"  • Tamaño: {len(df_high_weight):,} observaciones")
print(f"  • Rango fnlwgt: {df_high_weight['fnlwgt'].min():,} - {df_high_weight['fnlwgt'].max():,}")
print(f"  • Peso promedio: {df_high_weight['fnlwgt'].mean():.0f}")

print(f"\nMuestra 2 (PESO BAJO):")
print(f"  • Tamaño: {len(df_low_weight):,} observaciones")
print(f"  • Rango fnlwgt: {df_low_weight['fnlwgt'].min():,} - {df_low_weight['fnlwgt'].max():,}")
print(f"  • Peso promedio: {df_low_weight['fnlwgt'].mean():.0f}")

# Variables para clustering
numeric_cols = ['age', 'capital-gain', 'capital-loss', 'hours-per-week', 'education-num']
categorical_cols = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']

print(f"\nVariables numéricas para clustering: {numeric_cols}")
print(f"Variables categóricas para clustering: {categorical_cols}")
print(f"Total de observaciones a analizar: {len(df_high_weight) + len(df_low_weight):,}")

# ============================================================================
# FUNCIÓN PARA CALCULAR MATRIZ DE GOWER
# ============================================================================

def calcular_matriz_gower(df_sample):
    """Calcula la matriz de distancias de Gower para una muestra"""
    
    # Mapa ordinal para education
    education_map = {
        'Preschool': 1, '1st-4th': 2, '5th-6th': 3, '7th-8th': 4, '9th': 5,
        '10th': 6, '11th': 7, '12th': 8, 'HS-grad': 9, 'Some-college': 10,
        'Assoc-voc': 11, 'Assoc-acdm': 12, 'Bachelors': 13, 'Masters': 14,
        'Prof-school': 15, 'Doctorate': 16
    }
    
    # Normalizar education
    df_sample = df_sample.copy()
    df['education_rank'] = df['education'].map(education_map)
    min_rank = df['education_rank'].min()
    max_rank = df['education_rank'].max()
    ordinal_rank_range = max_rank - min_rank
    df_sample['education'] = (df_sample['education'].map(education_map) - min_rank) / ordinal_rank_range
    
    # Rangos numéricos
    ranges = {col: df[col].max() - df[col].min() for col in numeric_cols}
    
    # Convertir a arrays
    df_num = df_sample[numeric_cols].to_numpy(dtype=np.float64)
    df_cat = df_sample[categorical_cols].to_numpy(dtype=object)
    df_ord = df_sample[['education']].to_numpy(dtype=np.float64)
    
    # Calcular disimilitudes
    numeric_dissim = np.sum([
        np.abs(np.subtract.outer(df_num[:, i], df_num[:, i])) / ranges[col]
        for i, col in enumerate(numeric_cols) if ranges[col] > 0
    ], axis=0)
    
    categorical_dissim = np.sum([
        np.not_equal.outer(df_cat[:, i], df_cat[:, i]).astype(np.float64)
        for i in range(df_cat.shape[1])
    ], axis=0)
    
    ordinal_dissim = np.sum([
        np.abs(np.subtract.outer(df_ord[:, i], df_ord[:, i]))
        for i in range(df_ord.shape[1])
    ], axis=0)
    
    total_dissimilarity = numeric_dissim + categorical_dissim + ordinal_dissim
    
    # Pesos
    is_not_nan = ~df_sample.drop(columns=['education_rank'], errors='ignore').isna()
    weight_sum = np.dot(is_not_nan.astype(int), is_not_nan.astype(int).T)
    
    # Matriz final
    with np.errstate(divide='ignore', invalid='ignore'):
        gower_matrix = np.where(weight_sum > 0, total_dissimilarity / weight_sum, 0)
    
    return gower_matrix

# ============================================================================
# K-MEANS EN MUESTRA DE PESO ALTO
# ============================================================================

print("\n" + "="*70)
print("K-MEANS: MUESTRA DE PESO ALTO (Primeros 2500)")
print("="*70)

# Preparar datos de peso alto
X_high = df_high_weight[numeric_cols].copy()
scaler_high = StandardScaler()
X_high_scaled = scaler_high.fit_transform(X_high)

# Aplicar K-means con distintos valores de k
k_range = range(2, 11)
kmeans_high_results = {
    'k': [],
    'silhouette': [],
    'davies_bouldin': []
}

print("\nCalculando métricas para muestra de PESO ALTO...")
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_high_scaled)
    
    silhouette = silhouette_score(X_high_scaled, labels)
    davies_bouldin = davies_bouldin_score(X_high_scaled, labels)
    
    kmeans_high_results['k'].append(k)
    kmeans_high_results['silhouette'].append(silhouette)
    kmeans_high_results['davies_bouldin'].append(davies_bouldin)
    
    print(f"  k={k}: Silhouette={silhouette:.4f}, Davies-Bouldin={davies_bouldin:.4f}")

# Determinar k óptimo para muestra de peso alto
high_best_k_silhouette = k_range[np.argmax(kmeans_high_results['silhouette'])]
high_best_k_db = k_range[np.argmin(kmeans_high_results['davies_bouldin'])]

print(f"\nPESO ALTO - K óptimo según Silhouette: {high_best_k_silhouette}")
print(f"PESO ALTO - K óptimo según Davies-Bouldin: {high_best_k_db}")

# ============================================================================
# K-MEANS EN MUESTRA DE PESO BAJO
# ============================================================================

print("\n" + "="*70)
print("K-MEANS: MUESTRA DE PESO BAJO (Últimos 2500)")
print("="*70)

# Preparar datos de peso bajo
X_low = df_low_weight[numeric_cols].copy()
scaler_low = StandardScaler()
X_low_scaled = scaler_low.fit_transform(X_low)

# Aplicar K-means con distintos valores de k
kmeans_low_results = {
    'k': [],
    'silhouette': [],
    'davies_bouldin': []
}

print("\nCalculando métricas para muestra de PESO BAJO...")
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_low_scaled)
    
    silhouette = silhouette_score(X_low_scaled, labels)
    davies_bouldin = davies_bouldin_score(X_low_scaled, labels)
    
    kmeans_low_results['k'].append(k)
    kmeans_low_results['silhouette'].append(silhouette)
    kmeans_low_results['davies_bouldin'].append(davies_bouldin)
    
    print(f"  k={k}: Silhouette={silhouette:.4f}, Davies-Bouldin={davies_bouldin:.4f}")

# Determinar k óptimo para muestra de peso bajo
low_best_k_silhouette = k_range[np.argmax(kmeans_low_results['silhouette'])]
low_best_k_db = k_range[np.argmin(kmeans_low_results['davies_bouldin'])]

print(f"\nPESO BAJO - K óptimo según Silhouette: {low_best_k_silhouette}")
print(f"PESO BAJO - K óptimo según Davies-Bouldin: {low_best_k_db}")

# ============================================================================
# AGGLOMERATIVE CON GOWER EN MUESTRA DE PESO ALTO
# ============================================================================

print("\n" + "="*70)
print("AGGLOMERATIVE CON GOWER: MUESTRA DE PESO ALTO")
print("="*70)

print("\nCalculando matriz de Gower para muestra de PESO ALTO...")
gower_high = calcular_matriz_gower(df_high_weight)
np.fill_diagonal(gower_high, 0)

# Aplicar Agglomerative con distintos valores de k
agg_high_results = {
    'k': [],
    'silhouette': [],
    'davies_bouldin': []
}

print("\nCalculando métricas para muestra de PESO ALTO...")
for k in k_range:
    agg = AgglomerativeClustering(n_clusters=k, metric='precomputed', linkage='average')
    labels = agg.fit_predict(gower_high)
    
    silhouette = silhouette_score(gower_high, labels, metric='precomputed')
    davies_bouldin = davies_bouldin_score(gower_high, labels)
    
    agg_high_results['k'].append(k)
    agg_high_results['silhouette'].append(silhouette)
    agg_high_results['davies_bouldin'].append(davies_bouldin)
    
    print(f"  k={k}: Silhouette={silhouette:.4f}, Davies-Bouldin={davies_bouldin:.4f}")

# Determinar k óptimo para muestra de peso alto
high_agg_best_k_silhouette = k_range[np.argmax(agg_high_results['silhouette'])]
high_agg_best_k_db = k_range[np.argmin(agg_high_results['davies_bouldin'])]

print(f"\nPESO ALTO - K óptimo según Silhouette: {high_agg_best_k_silhouette}")
print(f"PESO ALTO - K óptimo según Davies-Bouldin: {high_agg_best_k_db}")

# ============================================================================
# AGGLOMERATIVE CON GOWER EN MUESTRA DE PESO BAJO
# ============================================================================

print("\n" + "="*70)
print("AGGLOMERATIVE CON GOWER: MUESTRA DE PESO BAJO")
print("="*70)

print("\nCalculando matriz de Gower para muestra de PESO BAJO...")
gower_low = calcular_matriz_gower(df_low_weight)
np.fill_diagonal(gower_low, 0)

# Aplicar Agglomerative con distintos valores de k
agg_low_results = {
    'k': [],
    'silhouette': [],
    'davies_bouldin': []
}

print("\nCalculando métricas para muestra de PESO BAJO...")
for k in k_range:
    agg = AgglomerativeClustering(n_clusters=k, metric='precomputed', linkage='average')
    labels = agg.fit_predict(gower_low)
    
    silhouette = silhouette_score(gower_low, labels, metric='precomputed')
    davies_bouldin = davies_bouldin_score(gower_low, labels)
    
    agg_low_results['k'].append(k)
    agg_low_results['silhouette'].append(silhouette)
    agg_low_results['davies_bouldin'].append(davies_bouldin)
    
    print(f"  k={k}: Silhouette={silhouette:.4f}, Davies-Bouldin={davies_bouldin:.4f}")

# Determinar k óptimo para muestra de peso bajo
low_agg_best_k_silhouette = k_range[np.argmax(agg_low_results['silhouette'])]
low_agg_best_k_db = k_range[np.argmin(agg_low_results['davies_bouldin'])]

print(f"\nPESO BAJO - K óptimo según Silhouette: {low_agg_best_k_silhouette}")
print(f"PESO BAJO - K óptimo según Davies-Bouldin: {low_agg_best_k_db}")


EJERCICIO 6: EVALUACIÓN DE CLUSTERS POR PESO

Dataset completo: 48,842 observaciones

----------------------------------------------------------------------
CREANDO MUESTRAS POR PESO (fnlwgt)
----------------------------------------------------------------------

Muestra 1 (PESO ALTO):
  • Tamaño: 2,500 observaciones
  • Rango fnlwgt: 377,798 - 1,490,400
  • Peso promedio: 465446

Muestra 2 (PESO BAJO):
  • Tamaño: 2,500 observaciones
  • Rango fnlwgt: 12,285 - 40,190
  • Peso promedio: 31722

Variables numéricas para clustering: ['age', 'capital-gain', 'capital-loss', 'hours-per-week', 'education-num']
Variables categóricas para clustering: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']
Total de observaciones a analizar: 5,000

K-MEANS: MUESTRA DE PESO ALTO (Primeros 2500)

Calculando métricas para muestra de PESO ALTO...
  k=2: Silhouette=0.6049, Davies-Bouldin=0.6644
  k=3: Silhouette=0.6182, Davies-Bouldin=0.5131
  k=4: S

## Paso 2: Comparación entre muestras de Peso Alto vs Peso Bajo

**Objetivo:** Analizar si hay diferencias significativas en el clustering entre personas con mayor vs menor peso.


In [2]:
# ============================================================================
# COMPARACIÓN ENTRE MUESTRAS DE PESO ALTO vs PESO BAJO
# ============================================================================

print("\n" + "="*70)
print("COMPARACIÓN: PESO ALTO vs PESO BAJO")
print("="*70)

# Crear tabla comparativa de métricas para K-means
kmeans_comparison_df = pd.DataFrame({
    'k': kmeans_high_results['k'],
    'K-means Silhouette Peso Alto': kmeans_high_results['silhouette'],
    'K-means Silhouette Peso Bajo': kmeans_low_results['silhouette'],
    'K-means Davies-Bouldin Peso Alto': kmeans_high_results['davies_bouldin'],
    'K-means Davies-Bouldin Peso Bajo': kmeans_low_results['davies_bouldin']
})

print("\nTabla Comparativa de Métricas - K-means:")
print(kmeans_comparison_df.to_string(index=False))

# Crear tabla comparativa de métricas para Agglomerative
agg_comparison_df = pd.DataFrame({
    'k': agg_high_results['k'],
    'Agglomerative Silhouette Peso Alto': agg_high_results['silhouette'],
    'Agglomerative Silhouette Peso Bajo': agg_low_results['silhouette'],
    'Agglomerative Davies-Bouldin Peso Alto': agg_high_results['davies_bouldin'],
    'Agglomerative Davies-Bouldin Peso Bajo': agg_low_results['davies_bouldin']
})

print("\nTabla Comparativa de Métricas - Agglomerative:")
print(agg_comparison_df.to_string(index=False))

# Calcular diferencias promedio para K-means
kmeans_avg_silhouette_diff = np.mean(np.array(kmeans_high_results['silhouette']) - np.array(kmeans_low_results['silhouette']))
kmeans_avg_db_diff = np.mean(np.array(kmeans_high_results['davies_bouldin']) - np.array(kmeans_low_results['davies_bouldin']))

# Calcular diferencias promedio para Agglomerative
agg_avg_silhouette_diff = np.mean(np.array(agg_high_results['silhouette']) - np.array(agg_low_results['silhouette']))
agg_avg_db_diff = np.mean(np.array(agg_high_results['davies_bouldin']) - np.array(agg_low_results['davies_bouldin']))

print(f"\n" + "="*70)
print("ANÁLISIS DE DIFERENCIAS")
print("="*70)

print(f"\nDiferencias promedio - K-means:")
print(f"  Silhouette (Peso Alto - Peso Bajo): {kmeans_avg_silhouette_diff:.4f}")
print(f"  Davies-Bouldin (Peso Alto - Peso Bajo): {kmeans_avg_db_diff:.4f}")

print(f"\nDiferencias promedio - Agglomerative:")
print(f"  Silhouette (Peso Alto - Peso Bajo): {agg_avg_silhouette_diff:.4f}")
print(f"  Davies-Bouldin (Peso Alto - Peso Bajo): {agg_avg_db_diff:.4f}")

# Comparar k óptimos
print(f"\n" + "-"*70)
print("COMPARACIÓN DE K ÓPTIMOS")
print("-"*70)

print(f"\nK-means - K óptimo según Silhouette:")
print(f"  • Peso Alto: k = {high_best_k_silhouette} (score: {max(kmeans_high_results['silhouette']):.4f})")
print(f"  • Peso Bajo: k = {low_best_k_silhouette} (score: {max(kmeans_low_results['silhouette']):.4f})")

print(f"\nK-means - K óptimo según Davies-Bouldin:")
print(f"  • Peso Alto: k = {high_best_k_db} (score: {min(kmeans_high_results['davies_bouldin']):.4f})")
print(f"  • Peso Bajo: k = {low_best_k_db} (score: {min(kmeans_low_results['davies_bouldin']):.4f})")

print(f"\nAgglomerative - K óptimo según Silhouette:")
print(f"  • Peso Alto: k = {high_agg_best_k_silhouette} (score: {max(agg_high_results['silhouette']):.4f})")
print(f"  • Peso Bajo: k = {low_agg_best_k_silhouette} (score: {max(agg_low_results['silhouette']):.4f})")

print(f"\nAgglomerative - K óptimo según Davies-Bouldin:")
print(f"  • Peso Alto: k = {high_agg_best_k_db} (score: {min(agg_high_results['davies_bouldin']):.4f})")
print(f"  • Peso Bajo: k = {low_agg_best_k_db} (score: {min(agg_low_results['davies_bouldin']):.4f})")

# Verificar consistencia para K-means
kmeans_silhouette_consistent = high_best_k_silhouette == low_best_k_silhouette
kmeans_db_consistent = high_best_k_db == low_best_k_db

# Verificar consistencia para Agglomerative
agg_silhouette_consistent = high_agg_best_k_silhouette == low_agg_best_k_silhouette
agg_db_consistent = high_agg_best_k_db == low_agg_best_k_db

print(f"\nConsistencia de k óptimo - K-means:")
print(f"  • Silhouette: {'CONSISTENTE' if kmeans_silhouette_consistent else 'INCONSISTENTE'}")
print(f"  • Davies-Bouldin: {'CONSISTENTE' if kmeans_db_consistent else 'INCONSISTENTE'}")

print(f"\nConsistencia de k óptimo - Agglomerative:")
print(f"  • Silhouette: {'CONSISTENTE' if agg_silhouette_consistent else 'INCONSISTENTE'}")
print(f"  • Davies-Bouldin: {'CONSISTENTE' if agg_db_consistent else 'INCONSISTENTE'}")

# Comparación entre algoritmos
print(f"\n" + "-"*70)
print("COMPARACIÓN ENTRE ALGORITMOS")
print("-"*70)

print(f"\nK-means vs Agglomerative - Peso Alto:")
print(f"  • K-means Silhouette óptimo: k = {high_best_k_silhouette} ({max(kmeans_high_results['silhouette']):.4f})")
print(f"  • Agglomerative Silhouette óptimo: k = {high_agg_best_k_silhouette} ({max(agg_high_results['silhouette']):.4f})")

print(f"\nK-means vs Agglomerative - Peso Bajo:")
print(f"  • K-means Silhouette óptimo: k = {low_best_k_silhouette} ({max(kmeans_low_results['silhouette']):.4f})")
print(f"  • Agglomerative Silhouette óptimo: k = {low_agg_best_k_silhouette} ({max(agg_low_results['silhouette']):.4f})")



COMPARACIÓN: PESO ALTO vs PESO BAJO

Tabla Comparativa de Métricas - K-means:
 k  K-means Silhouette Peso Alto  K-means Silhouette Peso Bajo  K-means Davies-Bouldin Peso Alto  K-means Davies-Bouldin Peso Bajo
 2                        0.6049                        0.5750                            0.6644                            0.6573
 3                        0.6182                        0.5884                            0.5131                            0.4932
 4                        0.2704                        0.2988                            0.9827                            0.9325
 5                        0.2853                        0.2944                            0.9394                            0.9256
 6                        0.3004                        0.3274                            0.9500                            0.8347
 7                        0.3137                        0.2823                            0.9315                            0.9102
 8  

## Paso 3: Análisis de resultados

**Objetivo:** Analizar los resultados obtenidos con ambos algoritmos de clustering.


In [3]:
# ============================================================================
# ANÁLISIS SOCIODEMOGRÁFICO POR PESO Y CLUSTERING
# ============================================================================

print("\n" + "="*70)
print("ANÁLISIS SOCIODEMOGRÁFICO POR PESO Y CLUSTERING")
print("="*70)

# Aplicar clustering óptimo a ambas muestras para análisis sociodemográfico
print("\nAplicando clustering óptimo para análisis sociodemográfico...")

# K-means óptimo para peso alto
kmeans_high_optimal = KMeans(n_clusters=high_best_k_silhouette, random_state=42, n_init=10)
X_high_scaled = scaler_high.fit_transform(df_high_weight[numeric_cols])
df_high_weight['kmeans_cluster'] = kmeans_high_optimal.fit_predict(X_high_scaled)

# K-means óptimo para peso bajo
kmeans_low_optimal = KMeans(n_clusters=low_best_k_silhouette, random_state=42, n_init=10)
X_low_scaled = scaler_low.fit_transform(df_low_weight[numeric_cols])
df_low_weight['kmeans_cluster'] = kmeans_low_optimal.fit_predict(X_low_scaled)

# Agglomerative óptimo para peso alto
agg_high_optimal = AgglomerativeClustering(n_clusters=high_agg_best_k_silhouette, metric='precomputed', linkage='average')
df_high_weight['agg_cluster'] = agg_high_optimal.fit_predict(gower_high)

# Agglomerative óptimo para peso bajo
agg_low_optimal = AgglomerativeClustering(n_clusters=low_agg_best_k_silhouette, metric='precomputed', linkage='average')
df_low_weight['agg_cluster'] = agg_low_optimal.fit_predict(gower_low)

print("✓ Clustering aplicado a ambas muestras")

# 1. ANÁLISIS SOCIODEMOGRÁFICO - MUESTRA PESO ALTO
print("\n" + "="*60)
print("1. ANÁLISIS SOCIODEMOGRÁFICO - MUESTRA PESO ALTO")
print("="*60)

print("\n--- K-MEANS CLUSTERING ---")
print("Distribución de ingresos por cluster (K-means):")
income_kmeans_high = pd.crosstab(df_high_weight['kmeans_cluster'], df_high_weight['income'], normalize='index') * 100
print(income_kmeans_high.round(1))

print("\nPerfiles sociodemográficos por cluster (K-means):")
for cluster in range(high_best_k_silhouette):
    cluster_data = df_high_weight[df_high_weight['kmeans_cluster'] == cluster]
    if len(cluster_data) > 0:
        high_income_pct = (cluster_data['income'] == '>50K').mean() * 100
        avg_age = cluster_data['age'].mean()
        male_pct = (cluster_data['sex'] == 'Male').mean() * 100
        main_country = cluster_data['native-country'].mode().iloc[0] if len(cluster_data) > 0 else "N/A"
        
        print(f"\nCluster {cluster}:")
        print(f"  - Ingresos altos: {high_income_pct:.1f}%")
        print(f"  - Edad promedio: {avg_age:.1f} años")
        print(f"  - Género masculino: {male_pct:.1f}%")
        

print("\n--- AGGLOMERATIVE CLUSTERING ---")
print("Distribución de ingresos por cluster (Agglomerative):")
income_agg_high = pd.crosstab(df_high_weight['agg_cluster'], df_high_weight['income'], normalize='index') * 100
print(income_agg_high.round(1))

# 2. ANÁLISIS SOCIODEMOGRÁFICO - MUESTRA PESO BAJO
print("\n" + "="*60)
print("2. ANÁLISIS SOCIODEMOGRÁFICO - MUESTRA PESO BAJO")
print("="*60)

print("\n--- K-MEANS CLUSTERING ---")
print("Distribución de ingresos por cluster (K-means):")
income_kmeans_low = pd.crosstab(df_low_weight['kmeans_cluster'], df_low_weight['income'], normalize='index') * 100
print(income_kmeans_low.round(1))

print("\nPerfiles sociodemográficos por cluster (K-means):")
for cluster in range(low_best_k_silhouette):
    cluster_data = df_low_weight[df_low_weight['kmeans_cluster'] == cluster]
    if len(cluster_data) > 0:
        high_income_pct = (cluster_data['income'] == '>50K').mean() * 100
        avg_age = cluster_data['age'].mean()
        male_pct = (cluster_data['sex'] == 'Male').mean() * 100
        main_country = cluster_data['native-country'].mode().iloc[0] if len(cluster_data) > 0 else "N/A"
        
        print(f"\nCluster {cluster}:")
        print(f"  - Ingresos altos: {high_income_pct:.1f}%")
        print(f"  - Edad promedio: {avg_age:.1f} años")
        print(f"  - Género masculino: {male_pct:.1f}%")
        

print("\n--- AGGLOMERATIVE CLUSTERING ---")
print("Distribución de ingresos por cluster (Agglomerative):")
income_agg_low = pd.crosstab(df_low_weight['agg_cluster'], df_low_weight['income'], normalize='index') * 100
print(income_agg_low.round(1))

# 3. COMPARACIÓN SOCIODEMOGRÁFICA ENTRE MUESTRAS
print("\n" + "="*60)
print("3. COMPARACIÓN SOCIODEMOGRÁFICA ENTRE MUESTRAS")
print("="*60)

print("\nDistribución general de ingresos:")
print(f"Peso Alto: {(df_high_weight['income'] == '>50K').mean()*100:.1f}% ingresos altos")
print(f"Peso Bajo: {(df_low_weight['income'] == '>50K').mean()*100:.1f}% ingresos altos")

print("\nDistribución por género:")
print("Peso Alto:")
gender_high = pd.crosstab(df_high_weight['sex'], df_high_weight['income'], normalize='index') * 100
print(gender_high.round(1))

print("\nPeso Bajo:")
gender_low = pd.crosstab(df_low_weight['sex'], df_low_weight['income'], normalize='index') * 100
print(gender_low.round(1))

print("\nDistribución por edad:")
print(f"Peso Alto - Edad promedio: {df_high_weight['age'].mean():.1f} años")
print(f"Peso Bajo - Edad promedio: {df_low_weight['age'].mean():.1f} años")

# 4. CONCLUSIONES SOCIODEMOGRÁFICAS
print("\n" + "="*60)
print("4. CONCLUSIONES SOCIODEMOGRÁFICAS")
print("="*60)

print("\nHallazgos principales:")
print("-" * 30)

# Análisis de consistencia general
kmeans_total_consistent = kmeans_silhouette_consistent and kmeans_db_consistent
agg_total_consistent = agg_silhouette_consistent and agg_db_consistent

print(f"\n1. CONSISTENCIA DE CLUSTERING:")
print(f"   • K-means: {'CONSISTENTE' if kmeans_total_consistent else 'INCONSISTENTE'}")
print(f"   • Agglomerative: {'CONSISTENTE' if agg_total_consistent else 'INCONSISTENTE'}")

print(f"\n2. IMPACTO DEL PESO EN CARACTERÍSTICAS SOCIODEMOGRÁFICAS:")
income_diff = (df_high_weight['income'] == '>50K').mean() - (df_low_weight['income'] == '>50K').mean()
age_diff = df_high_weight['age'].mean() - df_low_weight['age'].mean()
male_diff_high = (df_high_weight['sex'] == 'Male').mean() - (df_low_weight['sex'] == 'Male').mean()

print(f"   • Diferencia en ingresos altos: {income_diff*100:+.1f}%")
print(f"   • Diferencia en edad promedio: {age_diff:+.1f} años")
print(f"   • Diferencia en proporción masculina: {male_diff_high*100:+.1f}%")

print(f"\n3. PATRONES SOCIODEMOGRÁFICOS IDENTIFICADOS:")
print(f"   • K-means sugiere k={high_best_k_silhouette} para peso alto y k={low_best_k_silhouette} para peso bajo")
print(f"   • Agglomerative sugiere k={high_agg_best_k_silhouette} para peso alto y k={low_agg_best_k_silhouette} para peso bajo")

print(f"\n" + "="*70)
print("="*70)



ANÁLISIS SOCIODEMOGRÁFICO POR PESO Y CLUSTERING

Aplicando clustering óptimo para análisis sociodemográfico...
✓ Clustering aplicado a ambas muestras

1. ANÁLISIS SOCIODEMOGRÁFICO - MUESTRA PESO ALTO

--- K-MEANS CLUSTERING ---
Distribución de ingresos por cluster (K-means):
income          <=50K   >50K
kmeans_cluster              
0                 0.0  100.0
1                79.3   20.7
2                53.7   46.3

Perfiles sociodemográficos por cluster (K-means):

Cluster 0:
  - Ingresos altos: 100.0%
  - Edad promedio: 44.0 años
  - Género masculino: 75.0%

Cluster 1:
  - Ingresos altos: 20.7%
  - Edad promedio: 35.3 años
  - Género masculino: 71.3%

Cluster 2:
  - Ingresos altos: 46.3%
  - Edad promedio: 40.4 años
  - Género masculino: 78.9%

--- AGGLOMERATIVE CLUSTERING ---
Distribución de ingresos por cluster (Agglomerative):
income       <=50K  >50K
agg_cluster             
0            100.0   0.0
1             77.9  22.1

2. ANÁLISIS SOCIODEMOGRÁFICO - MUESTRA PESO BAJO

--